# System Specifications

In [1]:
!lsb_release -a
print('========================================================py==========================================================')
!python --version
print('========================================================gpu=========================================================')
!nvcc --version
!nvidia-smi
print('========================================================cpu=========================================================')
!lscpu | grep -E 'Architecture|CPU\(s\):|Model name|Thread|Core'

No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.4 LTS
Release:	22.04
Codename:	jammy
========================================================py==========================================================
Python 3.11.13
========================================================gpu=========================================================
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
Tue Jun 23 10:25:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf         

# Initialize Notebook

In [2]:
# @title 0: Initialize notebook

from IPython.display import clear_output
from IPython.display import display as i_display

clear_output()

# Installation

In [3]:
%cp -i -r /kaggle/input/datasets/kerneax/voxcpm-data/VoxCPM /kaggle/working

In [4]:
%cd /kaggle/working/VoxCPM
!pip install .

import os
os.environ["MODELSCOPE_CACHE"] = "/kaggle/working/VoxCPM"
clear_output()

# Resolve Dependencies

In [5]:
# 注意：生成音频报错，当前 kaggle py默认3.12，测试使用3.11，以下依赖重装
!pip install torch==2.6.0+cu126 torchvision==0.21.0+cu126 torchaudio==2.6.0+cu126 --extra-index-url https://download.pytorch.org/whl/cu126
!pip show torch torchvision torchaudio
clear_output()

# Frp Configuration

In [10]:
%cd /kaggle/working

import random
import string

chars = string.ascii_letters + string.digits

FRP_NAME = ''.join(random.choice(chars) for _ in range(16))
FRP_SERVER_ADDR = 'XXX_GLOBAL_FRP_SERVER_ADDR_XXX'
FRP_AUTH_TOKEN = 'XXX_GLOBAL_FRP_AUTH_TOKEN_XXX'
FRP_PORT = "XXX_GLOBAL_FRP_PORT_XXX"

#Download Frp
!wget https://github.com/fatedier/frp/releases/download/v0.68.1/frp_0.68.1_linux_amd64.tar.gz
!tar -xzf frp_0.68.1_linux_amd64.tar.gz

#Frp client config
frpc_config = f"""
serverAddr = "{ FRP_SERVER_ADDR }"
serverPort = 7000
auth.token = "{ FRP_AUTH_TOKEN }"
loginFailExit = false

[[proxies]]
name = "{ FRP_NAME }"
type = "tcp"
localPort = 8808
remotePort = { FRP_PORT }
"""
with open('frp_0.68.1_linux_amd64/frpc.toml', 'w') as f:
    f.write(frpc_config)

clear_output()

# Launcher

In [7]:
!curl -L -o /kaggle/working/VoxCPM/app_v2_1.py "https://raw.githubusercontent.com/kerneax/voxcpm-tester/refs/heads/main/app_v2_1.py"

clear_output()

# Tester Script

In [8]:
%cd /kaggle/working

#Tester Script
tester_script = """
import requests
import time

# Define the API endpoint
url = "http://127.0.0.1:8808/gradio_api/call/generate"

# Define the payload data exactly as in the curl command
payload = {
    "data": [
        "Hello World",
        "A warm young man",
        None,
        False,
        "",
        2,
        False,
        False,
        10
    ]
}

# Make the POST request
try:
    print("Local API Test will be executed in 3 miniutes.")
    time.sleep(180)
    response = requests.post(url, json=payload)
    
    # Check if the request was successful
    response.raise_for_status()
    
    # Print the JSON response
    print("=========Local API Test Success===========")
    print(response.json())

except requests.exceptions.RequestException as e:
    print("=========Local API Test Failed===========")
    print(f"An error occurred: {e}")
"""
with open('VoxCPM/tester_script.py', 'w') as f:
    f.write(tester_script)

/kaggle/working


# Run

In [ ]:
# 项目目录
%cd /kaggle/working/VoxCPM

# 启动
!TORCHDYNAMO_DISABLE=1 python app_v2_1.py --model-id models/openbmb__VoxCPM2 & python tester_script.py & /kaggle/working/frp_0.68.1_linux_amd64/frpc -c /kaggle/working/frp_0.68.1_linux_amd64/frpc.toml

/kaggle/working/VoxCPM
2026-06-23 10:33:36.191 [I] [sub/root.go:201] start frpc service for config file [/kaggle/working/frp_0.68.1_linux_amd64/frpc.toml] with aggregated configuration
2026-06-23 10:33:36.191 [I] [client/service.go:378] try to connect to server...
Local API Test will be executed in 3 miniutes.
2026-06-23 10:33:36.700 [I] [client/service.go:370] [4842438cce9070a5] login to server success, get run id [4842438cce9070a5]
2026-06-23 10:33:36.700 [I] [proxy/proxy_manager.go:180] [4842438cce9070a5] proxy added: [D677maGjJ7LN1I45]
2026-06-23 10:33:36.868 [I] [client/control.go:176] [4842438cce9070a5] [D677maGjJ7LN1I45] start proxy success
funasr version: 1.3.13.
* Running on local URL:  http://0.0.0.0:8808
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://f02e8092f4b9f01258.gradio.live

Thi